<a href="https://colab.research.google.com/github/ThiagoPoppe/lamir2026-summer-code/blob/main/fingerprinting_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔬️ Audio Fingerprinting Experiments: Analyzing Fingerprint Robustness to Audio Degradation

This notebook explores the robustness of the fingerprinting algorithm against different audio modifications using [GTZAN](https://www.cs.cmu.edu/~gtzan/work/pubs/tsap02gtzan.pdf), a well-known dataset in the field of Music Information Retrieval (MIR).

Although this dataset was not originally designed for audio fingerprinting, but rather for music genre classification, it serves as a useful resource, providing 30-second audio samples from different music genres. This allows us to conduct experiments aimed at investigating the limitations of the fingerprinting algorithm proposed by [Wang em 2003](https://www.ee.columbia.edu/~dpwe/papers/Wang03-shazam.pdf)..

In [2]:
# Required to run the audiomentations library used to augment audio samples
!pip install audiomentations[extras] --quiet

In [4]:
import os
import bisect
import librosa
import kagglehub
import urllib.request

import numpy as np
import pandas as pd
import ipywidgets as widgets
import IPython.display as ipd
import scipy.ndimage as ndimage
import matplotlib.pyplot as plt

from collections import defaultdict
from dataclasses import dataclass, replace
from typing import Any, Dict, Literal, List, Tuple, Optional

from audiomentations import (
    Compose,
    PitchShift,
    TimeStretch,
    BandPassFilter,
    ClippingDistortion,
    Mp3Compression,
    AddGaussianSNR
)

from google.colab import output
output.enable_custom_widget_manager()

## 1. The GTZAN Dataset

Criado por [George Tzanetakis (2002)](https://www.cs.cmu.edu/~gtzan/work/pubs/tsap02gtzan.pdf), o conjunto de dados GTZAN é usado amplamente para a avaliação de detecção automática de gênero musical. Por conta disso, a base contém uma distribuição igualitária de 1000 clipes de 30 segundos cada em 10 gêneros musicais.

Na célula abaixo nós carregamos esse dataset e exibimos algumas estatísticas do seu conteúdo.

> É importante notar que por mais que o dataset é extremamente balanceado, há algumas faixas duplicadas e um arquivo em específico de Jazz que resulta em um erro ao carregarmos. Por esse motivo, excluímos esse arquivo, mas deixamos o restante da base de dados intacta. Caso você tenha interesse em uma análise mais aprofundada dessa base de dados, juntamente com suas limitações, críticas, entre outros, é recomendado a leitura do artigo de [Bob L. Sturm (2024)](https://arxiv.org/pdf/1306.1461).



In [3]:
dataset_path = kagglehub.dataset_download("andradaolteanu/gtzan-dataset-music-genre-classification")
audio_dir = os.path.join(dataset_path, "Data", "genres_original")

print('Path to audio files:', audio_dir)

Using Colab cache for faster access to the 'gtzan-dataset-music-genre-classification' dataset.
Path to audio files: /kaggle/input/gtzan-dataset-music-genre-classification/Data/genres_original


O conjunto de dados do GTZAN não especifica o nome das músicas nem o artista associado. Para isso, iremos fazer um cruzamento dos dados com o [repositório do Bob L. Sturm](https://github.com/boblsturm/GTZAN/tree/main) que contém alguns metadados como quais arquivos devem ser usados para o treinamento e avaliação de modelos e, em particular, no que estamos interessados, um mapeamento de nome do arquivo para o nome da música e artista, respectivamente.

In [5]:
index_url = "https://raw.githubusercontent.com/boblsturm/GTZAN/main/index.txt"
metadata_map = {}

with urllib.request.urlopen(index_url) as response:
    for raw_line in response:
        line = raw_line.decode("utf-8").strip()
        if not line or line.startswith("##"):
            continue

        parts = [p.strip() for p in line.split(":::")]
        filename = parts[0]
        artist = parts[1] if len(parts) > 1 and parts[1] else "Unknown"
        title = parts[2] if len(parts) > 2 and parts[2] else "Unknown"

        metadata_map[filename] = {
            "artist": artist,
            "title": title,
            "fullname": f"{artist} - {title}" if artist != "Unknown" else filename
        }

Como mencionado anteriormente, o arquivo `jazz.00054.wav` aparenta estar corrimpido, não permitindo assim o seu carregamento. Então iremos remover ele do nosso catálogo e criar um `Dataframe` para deixar todas as informações necessárias organizadas.

In [7]:
CORRUPTED_FILES = {"jazz.00054.wav"}
records = []

for genre in sorted(os.listdir(audio_dir)):
    genre_path = os.path.join(audio_dir, genre)
    if not os.path.isdir(genre_path):
        continue

    for filename in sorted(os.listdir(genre_path)):
        if not filename.endswith(".wav") or filename in CORRUPTED_FILES:
            continue

        meta = metadata_map.get(filename, {"artist": "Unknown", "title": "Unknown", "fullname": filename})
        records.append({
            "track_id": os.path.splitext(filename)[0],  # e.g., "blues.00000"
            "filename": filename,
            "genre": genre,
            "artist": meta["artist"],
            "title": meta["title"],
            "display_name": meta["fullname"],
            "filepath": os.path.join(genre_path, filename)
        })

catalog_df = pd.DataFrame(records)

In [8]:
print(f"Total valid tracks indexed: {len(catalog_df)}")
catalog_df

Total valid tracks indexed: 999


,track_id,filename,genre,artist,title,display_name,filepath
0,blues.00000,blues.00000.wav,blues,John Lee Hooker,"One Bourbon, One Scotch And One Beer","John Lee Hooker - One Bourbon, One Scotch And ...",/kaggle/input/gtzan-dataset-music-genre-classi...
1,blues.00001,blues.00001.wav,blues,John Lee Hooker,I'm In The Mood For Love,John Lee Hooker - I'm In The Mood For Love,/kaggle/input/gtzan-dataset-music-genre-classi...
2,blues.00002,blues.00002.wav,blues,John Lee Hooker,Think Twice Before You Go,John Lee Hooker - Think Twice Before You Go,/kaggle/input/gtzan-dataset-music-genre-classi...
3,blues.00003,blues.00003.wav,blues,John Lee Hooker,I'm Bad Like Jesse James,John Lee Hooker - I'm Bad Like Jesse James,/kaggle/input/gtzan-dataset-music-genre-classi...
4,blues.00004,blues.00004.wav,blues,John Lee Hooker,Walkin' the Boogie,John Lee Hooker - Walkin' the Boogie,/kaggle/input/gtzan-dataset-music-genre-classi...
...,...,...,...,...,...,...,...
994,rock.00095,rock.00095.wav,rock,The Stone Roses,Bye Bye Bad Man,The Stone Roses - Bye Bye Bad Man,/kaggle/input/gtzan-dataset-music-genre-classi...
995,rock.00096,rock.00096.wav,rock,The Stone Roses,(Song For My) Sugar Spun Sister,The Stone Roses - (Song For My) Sugar Spun Sister,/kaggle/input/gtzan-dataset-music-genre-classi...
996,rock.00097,rock.00097.wav,rock,The Stone Roses,Made Of Stone,The Stone Roses - Made Of Stone,/kaggle/input/gtzan-dataset-music-genre-classi...
997,rock.00098,rock.00098.wav,rock,The Stone Roses,Shoot You Down,The Stone Roses - Shoot You Down,/kaggle/input/gtzan-dataset-music-genre-classi...


Com isso, podemos fazer estatísticas básicas, como por exemplo verificar quais são os top 10 artistas que mais aparecem no catálogo de trechos de músicas.

In [13]:
@widgets.interact(
    genre=widgets.Dropdown(options=sorted(catalog_df.genre.unique()) + ['All'], description='Genre')
)
def plot_top10_artists_per_genre(genre):
    if genre == 'All':
        top10_artists = catalog_df['artist'].value_counts().head(10)
    else:
        genre_catalog_df = catalog_df[catalog_df.genre == genre]
        top10_artists = genre_catalog_df['artist'].value_counts().head(10)

    plt.title(f'Top 10 most common artists in the "{genre}" genre')
    plt.barh(top10_artists.index, top10_artists.values)
    plt.xlabel('Number of songs')
    plt.ylabel('Artist')
    plt.show()

interactive(children=(Dropdown(description='Genre', options=('blues', 'classical', 'country', 'disco', 'hiphop…

Nós também podemos filtrar o catálogo por um artista específico, por exemplo `Queen` e carregar um clipe de áudio da música `Bohemian Rhapsody` por exemplo.

In [17]:
all_artists = sorted(catalog_df.artist.dropna().unique())

artist_box = widgets.Combobox(
    options=all_artists,
    placeholder='Type here...',
    description='Artist:',
    ensure_option=True,
    layout=widgets.Layout(width='400px'),
)

SONG_DROPDOWN_PLACEHOLDER = ('Select a song...', None)
song_dropdown = widgets.Dropdown(
    options=[SONG_DROPDOWN_PLACEHOLDER],
    description='Song:',
    disabled=True,
    layout=widgets.Layout(width='400px'),
)

out = widgets.Output()


def on_artist_change(change):
    artist = change['new']
    out.clear_output()
    if artist in all_artists:
        songs = catalog_df[catalog_df.artist == artist]
        song_dropdown.options = [SONG_DROPDOWN_PLACEHOLDER] + list(zip(songs.title, songs.filepath))
        song_dropdown.disabled = False
    else:
        song_dropdown.options = [SONG_DROPDOWN_PLACEHOLDER]
        song_dropdown.disabled = True


def on_song_change(change):
    filepath = change['new']
    out.clear_output()

    if filepath is None:
        return

    with out:
        print('Loading file:', filepath)
        wave, sr = librosa.load(filepath, sr=None)
        print('Loaded sampling rate:', sr)
        ipd.display(ipd.Audio(wave, rate=sr))


artist_box.observe(on_artist_change, names='value')
song_dropdown.observe(on_song_change, names='value')

display(widgets.VBox([artist_box, song_dropdown, out]))

Agora que nós introduzimos a base de dados que iremos utilizar, vamos para os experimentos.

## 2. Experiments

### 2.1. Configuration dataclasses and useful functions

To focus this section only on the experiments itself, we provide a collection of useful functions that will help you guide your own experiments and take your own conclusions. Please, read the documentation of each function, as you will be asked to modify some of its parameters along this section.

In [21]:
@dataclass(frozen=True)
class AudioConfig:
    """
    Configuration parameters to load an audio file.

    Parameters
    ----------
    sample_rate : int, optional (default = None)
        Target sample rate to resample the audio. If None, the native sample rate is used.
    start : float, optional (default = 0.0)
        Time offset in seconds to start loading the audio. Must be between 0 and 30 seconds.
    duration : float, optional (default = None)
        Duration of the audio to load in seconds. Must not exceed 30 seconds.
        If None, the whole audio starting from `start` is loaded.
    """

    sample_rate: Optional[int] = None
    start: float = 0.0
    duration: Optional[float] = None


@dataclass(frozen=True)
class FeatureConfig:
    """
    Configuration parameters used to compute the time-frequency representation
    from which the peaks are extracted.

    Parameters
    ----------
    kind : {'stft', 'cqt', 'chroma'}, optional (default = 'stft')
        Type of representation to compute.
        - 'stft': magnitude of the Short-Time Fourier Transform, with linear
          frequency resolution.
        - 'cqt': magnitude of the Constant-Q Transform, with logarithmic
          frequency resolution of one bin per semitone, starting at C1
          (~32.7 Hz).
        - 'chroma': 12-bin pitch-class profile (one bin per semitone, octaves
          folded together), computed from the STFT.
    n_fft : int, optional (default = 2048)
        Length of the analysis window in samples. Larger values give finer
        frequency resolution but coarser time resolution. Used by 'stft' and
        'chroma'; ignored by 'cqt'.
    hop_length : int, optional (default = 512)
        Number of samples between consecutive frames. Smaller values give
        finer time resolution (and more frames). For 'cqt' it must be a
        multiple of 2**(n_octaves - 1), i.e. 64 for the default 84 bins.
    win_length : int, optional (default = None)
        Size of the window function in samples. Must not exceed `n_fft`.
        If None, `n_fft` is used. Used by 'stft' and 'chroma'; ignored by 'cqt'.
    num_bins : int, optional (default = None)
        Number of frequency bins to keep. If None, a default is chosen by `kind`:
        128 lowest-frequency bins for 'stft' and 84 semitones (7 octaves,
        C1 to B7) for 'cqt'. Ignored by 'chroma', which always has 12 bins.
    """

    kind: Literal["stft", "cqt", "chroma"] = "stft"
    n_fft: int = 2048
    hop_length: int = 512
    win_length: Optional[int] = None
    num_bins: Optional[int] = None


@dataclass(frozen=True)
class PeakConfig:
    """
    Configuration parameters used to build the constellation map with peak peaking.

    Parameters
    ----------
    freq_neighborhood : int, optional (default = 11)
        Height of the local window, in frequency bins (centered on the point,
        so a value of 11 looks 5 bins up and 5 bins down). Should be odd.
    time_neighborhood : int, optional (default = 5)
        Width of the local window, in time frames (centered on the point,
        so a value of 5 looks 2 frames left and 2 frames right). Should be odd.
    threshold : float, optional (default = 0.01)
        Minimum magnitude required to be considered a peak.
    """

    freq_neighborhood: int = 11
    time_neighborhood: int = 5
    threshold: float = 0.01


@dataclass(frozen=True)
class HashConfig:
    """
    Configuration parameters used to build the hashes based on the constellation maps.

    Parameters
    ----------
    target_zone_freq_size : int, optional (default = 31)
        Height of the target zone, in frequency bins (centered on the anchor's
        frequency, so a value of 31 looks 15 bins up and 15 bins down). Must be odd.
    target_zone_time_size : int, optional (default = 19)
        Width of the target zone, in time frames (centered `target_zone_time_delay`
        frames after the anchor, so a value of 19 spans 9 frames on each side of
        that center). Must be odd.
    target_zone_time_delay : int, optional (default = 20)
        Number of frames between the anchor and the center of the target zone.
        Must be larger than half of `target_zone_time_size`, so that every
        target lies strictly after the anchor.
    """

    target_zone_freq_size: int = 31
    target_zone_time_size: int = 19
    target_zone_time_delay: int = 20

    def __post_init__(self):
        if self.target_zone_freq_size % 2 == 0 or self.target_zone_time_size % 2 == 0:
            raise ValueError("target_zone_freq_size and target_zone_time_size must be odd.")

        if self.target_zone_time_delay <= self.target_zone_time_size // 2:
            raise ValueError(
                "target_zone_time_delay must exceed half of target_zone_time_size, "
                "otherwise targets can fall on or before the anchor."
            )

In [19]:
def load_audio(filepath: str, cfg: AudioConfig = AudioConfig()) -> tuple[np.ndarray, int]:
    """
    Load an audio file as a mono waveform.

    Parameters
    ----------
    filepath : str
        Path to the audio file. Any format supported by librosa/soundfile
        or audioread can be used.
    cfg : AudioConfig, optional (default = AudioConfig())
        Parameters controlling resampling and which excerpt is loaded.

    Returns
    -------
    wave : np.ndarray, shape (n_samples,)
        Mono waveform, as floating-point values in [-1, 1].
    sr : int
        Sample rate of `wave` in Hz. Equals `cfg.sample_rate` if it was given,
        otherwise the native sample rate of the file.

    Raises
    ------
    ValueError
        If `cfg.start` or `cfg.duration` are out of range.
    """

    if cfg.start < 0 or cfg.start > 30:
        raise ValueError(f'Invalid start time: {cfg.start}')

    if cfg.duration is not None and (cfg.duration < 0 or cfg.duration > 30):
        raise ValueError(f'Invalid duration time: {cfg.duration}')

    return librosa.load(filepath, sr=cfg.sample_rate, offset=cfg.start, duration=cfg.duration)

In [ ]:
def compute_spectrogram(wave: np.ndarray, sr: int, cfg: FeatureConfig = FeatureConfig()) -> np.ndarray:
    """
    Compute a non-negative time-frequency representation of a waveform.

    Parameters
    ----------
    wave : np.ndarray, shape (n_samples,)
        Mono waveform.
    sr : int
        Sample rate of `wave` in Hz.
    cfg : FeatureConfig, optional (default = FeatureConfig())
        Parameters defining the kind of representation and its resolution.

    Returns
    -------
    spectrogram : np.ndarray, shape (num_bins, n_frames)
        Magnitude spectrogram. `num_bins` is `cfg.num_bins` for 'stft' and 'cqt',
        and 12 for 'chroma'. Values are on the scale of the raw STFT magnitude
        for 'stft' and normalized for 'chroma', so thresholds are not directly
        comparable between the two.

    Raises
    ------
    ValueError
        If `cfg.kind` is not 'stft', 'cqt', or 'chroma'.
    """

    _DEFAULT_NUM_BINS = {
        "stft": 128,
        "cqt": 84,
    }

    stft_args = dict(n_fft=cfg.n_fft, hop_length=cfg.hop_length, win_length=cfg.win_length)
    num_bins = cfg.num_bins or _DEFAULT_NUM_BINS.get(cfg.kind)

    if cfg.kind == "stft":
        return np.abs(librosa.stft(wave, **stft_args))[:num_bins]
    if cfg.kind == "cqt":
        return np.abs(librosa.cqt(wave, sr=sr, hop_length=cfg.hop_length, n_bins=num_bins))
    if cfg.kind == "chroma":
        return librosa.feature.chroma_stft(y=wave, sr=sr, **stft_args)
    raise ValueError(
        f"Unknown feature kind: {cfg.kind!r}. Possible values are 'stft', 'cqt' or 'chroma'."
    )

In [ ]:
def find_peaks(spectrogram: np.ndarray, cfg: PeakConfig = PeakConfig()) -> np.ndarray:
    """
    Find the local maxima of a spectrogram that make up the constellation map.

    A point is a peak if it is the maximum of its local window (of size
    `freq_neighborhood` x `time_neighborhood`) and its magnitude is above
    `threshold`.

    Parameters
    ----------
    spectrogram : np.ndarray, shape (n_freq_bins, n_frames)
        Non-negative time-frequency representation, e.g. the output of
        `compute_spectrogram`.
    cfg : PeakConfig, optional (default = PeakConfig())
        Parameters defining the local window and the minimum peak magnitude.

    Returns
    -------
    peaks : np.ndarray of bool, shape (n_freq_bins, n_frames)
        Binary mask with the same shape as `spectrogram`, True where a peak
        was found. Use `np.argwhere(peaks)` to get (freq_bin, frame) pairs.
    """
    local_max = ndimage.maximum_filter(
        spectrogram, size=(cfg.freq_neighborhood, cfg.time_neighborhood)
    )
    return (spectrogram == local_max) & (spectrogram > cfg.threshold)

In [ ]:
def compute_constellation_map(
    filepath: str,
    audio_cfg: AudioConfig = AudioConfig(),
    features_cfg: FeatureConfig = FeatureConfig(),
    peaks_cfg: PeakConfig = PeakConfig(),
) -> np.ndarray:
    """
    Compute the constellation map of an audio file.

    The constellation map is a sparse binary time-frequency image that keeps
    only the most prominent spectral peaks. The pipeline is:
    load audio -> compute spectrogram -> pick local peaks.

    Parameters
    ----------
    filepath : str
        Path to the audio file.
    audio_cfg : AudioConfig, optional (default = AudioConfig())
        Parameters controlling how the audio is loaded.
    features_cfg : FeatureConfig, optional (default = FeatureConfig())
        Parameters controlling the time-frequency representation.
    peaks_cfg : PeakConfig, optional (default = PeakConfig())
        Parameters controlling peak picking.

    Returns
    -------
    constellation_map : np.ndarray of bool, shape (n_freq_bins, n_frames)
        True where a spectral peak was found. See `compute_spectrogram` for the
        meaning of the axes, and `find_peaks` for the peak criterion.

    See Also
    --------
    load_audio, compute_spectrogram, find_peaks :
        The individual steps, useful for inspecting intermediate results.
    """

    wave, sr = load_audio(filepath, audio_cfg)
    spectrogram = compute_spectrogram(wave, sr, features_cfg)
    return find_peaks(spectrogram, peaks_cfg)

In [ ]:
def compute_hashes(
    constellation_map: np.ndarray,
    sr: int,
    hop_length: Optional[int] = 512,
    metadata: Optional[Dict[str, Any]] = None,
    cfg: HashConfig = HashConfig()
) -> Dict[str, List[Dict[str, Any]]]:
    """
    Generate fingerprint hashes from a constellation map using anchor-target pairs.

    Every peak in the constellation map is treated as an anchor. For each anchor,
    a rectangular target zone is defined in the time-frequency plane, centered
    `cfg.target_zone_time_delay` frames after the anchor in time and on the anchor's
    frequency bin. Each peak that falls inside this zone is paired with the
    anchor to form a hash of the form ``"{anchor_freq}_{target_freq}_{timedelta}"``.

    Parameters
    ----------
    constellation_map : numpy.ndarray
        Boolean 2D array of shape ``(num_freq_bins, num_time_frames)``, where
        True marks a constellation peak.
    sr : int
        Sample rate in Hz of the audio the constellation map was computed from.
    hop_length : int
        Hop length in samples used to compute the spectrogram. Together with `sr`
        it converts frame indices to seconds (frames are assumed centered, as in
        librosa's defaults).
    metadata : dict of str to Any, optional (default = None)
        Additional information (e.g. song name or ID) stored alongside every hash
        occurrence. If None, an empty dictionary is used.
    cfg : HashConfig, optional (default = HashConfig())
        Parameters defining the target zone for hashing.

    Returns
    -------
    collections.defaultdict of str to list of dict
        Mapping from each hash string to the list of its occurrences. Each
        occurrence is a dictionary containing the entries from `metadata` plus
        ``'offset_in_seconds'`` (float), the time of the anchor peak.

    Notes
    -----
    Hashes are only comparable between constellation maps computed with the same
    `FeatureConfig`, since the meaning of a frequency bin depends on it.

    ``'offset_in_seconds'`` is relative to the start of the loaded excerpt
    (see `AudioConfig.start`).

    Peaks are sorted by time so that the candidate targets for each anchor can
    be located efficiently with binary search instead of scanning every peak.
    """

    if metadata is None:
        metadata = {}

    peak_freqs, peak_times = np.where(constellation_map)
    peaks = sorted(zip(peak_freqs, peak_times), key=lambda peak: peak[1])
    sorted_peak_times = [peak_time for _, peak_time in peaks]

    seconds_per_frame = hop_length / sr
    half_freq = cfg.target_zone_freq_size // 2
    half_time = cfg.target_zone_time_size // 2

    hashes = defaultdict(list)
    for anchor_freq, anchor_time in peaks:
        zone_min_freq = anchor_freq - half_freq
        zone_max_freq = anchor_freq + half_freq

        zone_center_time = anchor_time + cfg.target_zone_time_delay
        zone_min_time = zone_center_time - half_time
        zone_max_time = zone_center_time + half_time

        anchor_info = {**metadata, "offset_in_seconds": float(anchor_time * seconds_per_frame)}

        first_candidate_idx = bisect.bisect_left(sorted_peak_times, zone_min_time)
        last_candidate_idx = bisect.bisect_right(sorted_peak_times, zone_max_time)

        for target_freq, target_time in peaks[first_candidate_idx:last_candidate_idx]:
            if zone_min_freq <= target_freq <= zone_max_freq:
                time_delta = target_time - anchor_time
                hashes[f"{anchor_freq}_{target_freq}_{time_delta}"].append(anchor_info)

    return hashes

### 2.1. Fingerprint Accuracy with Query Size

### 2.2. Fingerprint Robustness to Spectrogram Parameters

### 2.3. Fingerprint Robustness to Audio Degradation

So far, we have only extracted the fingerprint of a pristine, well-behaved audio track. However, real-world queries, like the ones received by Shazam, are rarely perfect. This was especially true back in 2003 when the paper was published. At that time, the algorithm had to accurately identify songs recorded in noisy bars, captured through primitive flip-phone microphones, and heavily compressed over 2G cellular networks.

To test the robustness of our constellation maps, we will artificially degrade our audio query to simulate an early 2000s mobile phone recording. We will use the [audiomentations](https://github.com/iver56/audiomentations) Python library, which makes it easy to apply realistic audio distortions.

In [ ]:
zearly_mobile_pipeline = Compose([
    # 1. Simulate the narrow frequency band of old 2G cellular networks
    BandPassFilter(min_center_freq=1500.0, max_center_freq=2000.0, min_bandwidth_fraction=1.0, max_bandwidth_fraction=1.5, p=1.0),

    # 2. Apply heavy digital compression (early GSM networks were ~13 kbps)
    Mp3Compression(min_bitrate=8, max_bitrate=32, backend="fast-mp3-augment", p=1.0),

    # 3. Simulate a cheap microphone clipping
    ClippingDistortion(min_percentile_threshold=1, max_percentile_threshold=5, p=1.0),

    # 4. Add proportional static hiss (using SNR ensures it scales with track volume)
    AddGaussianSNR(min_snr_db=10.0, max_snr_db=15.0, p=1.0)
])

In [ ]:
print('Original Waveform')
ipd.display(ipd.Audio(data=wave, rate=sr))

print('Artificially Degraded Waveform')
degraded_wave = early_mobile_pipeline(samples=wave, sample_rate=sr)
ipd.display(ipd.Audio(data=degraded_wave, rate=sr))

In [ ]:
degraded_stft = np.abs(librosa.stft(degraded_wave))[:128, :]
degraded_constellation_map = create_constellation_map(degraded_stft)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.set_title('Constellation Map Robustness: Original vs. Artifically Degraded Recording')

img = ax.imshow(np.log1p(filtered_stft_mag), cmap='magma', origin='lower', aspect='auto')

original_peaks = np.where(constellation_map)
ax.scatter(original_peaks[1], original_peaks[0], color='red', marker='o', s=15, label='Original Peaks')

degraded_peaks = np.where(degraded_constellation_map)
ax.scatter(degraded_peaks[1], degraded_peaks[0], color='lightblue', marker='o', s=8, label='Degraded Peaks')

ax.set_xlabel('Spectrogram Frames')
ax.set_ylabel('Frequency Bins')
fig.legend(loc='upper right')
fig.tight_layout()
plt.show()

Looking at the plot above, you will notice that while some of the weaker peaks are swallowed by the noise and compression, the core "skeleton" of our constellation map survives intact. This highlights the genius of the algorithm: because it ultimately relies on the relative positioning of these points rather than their exact amplitudes, capturing just a subset of the original peaks is more than enough to yield a match.

### 2.4. Fingerprint Robustness to Audio Structure

However, noise and compression only alter the *energy* of the audio. What happens when a distortion alters the structural *coordinates* themselves, like if a DJ speeds up the track or shifts the pitch? Let's build a new pipeline to test the algorithm's limits.

In [ ]:
remix_pipeline = Compose([
    # 1. Speed up the track by 10% (alters the time coordinates)
    TimeStretch(min_rate=1.10, max_rate=1.10, p=1.0),

    # 2. Drop the pitch by exactly one semitone (alters the frequency coordinates)
    PitchShift(min_semitones=-1, max_semitones=-1, p=1.0)
])

In [ ]:
print('Original Waveform')
ipd.display(ipd.Audio(data=wave, rate=sr))

print('Remixed Pitch Waveform')
remixed_wave = remix_pipeline(samples=wave, sample_rate=sr)
ipd.display(ipd.Audio(data=remixed_wave, rate=sr))

In [ ]:
remixed_stft = np.abs(librosa.stft(remixed_wave))[:128, :]
remixed_constellation_map = create_constellation_map(remixed_stft)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.set_title('Constellation Map Robustness: Original vs. Remixed Recording')

img = ax.imshow(np.log1p(filtered_stft_mag), cmap='magma', origin='lower', aspect='auto')

original_peaks = np.where(constellation_map)
ax.scatter(original_peaks[1], original_peaks[0], color='red', marker='o', s=15, label='Original Peaks')

remixed_peaks = np.where(remixed_constellation_map)
ax.scatter(remixed_peaks[1], remixed_peaks[0], color='lightblue', marker='o', s=8, label='Remixed Peaks')

ax.set_xlabel('Spectrogram Frames')
ax.set_ylabel('Frequency Bins')
fig.legend(loc='upper right')
fig.tight_layout()
plt.show()

As you can see, almost no peaks survived our "remixed" pipeline. This reveals the primary weakness of this type of retrieval algorithm: it is designed strictly for *exact-audio matching*. It cannot handle tempo changes, pitch shifts, or cover versions.

However, you can still retrieve these modified tracks using alignment algorithms like [Dynamic Time Warping (DTW)](https://www.audiolabs-erlangen.de/resources/MIR/FMP/C7/C7S2_SubsequenceDTW.html), which are computationally much heavier than the fingerprinting approach explained here. If you want to explore this area further, the [Fundamentals of Music Processing (FMP) Notebooks](https://www.audiolabs-erlangen.de/resources/MIR/FMP/C7/C7S2_AudioMatching.html) by *Meinard Müller* are highly recommended.

## 3. Discussions